In [8]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import random

In [9]:
phrases = [
'Rise with courage, chase dreams, and never give up.',

'Every setback builds strength, keep going forward.',

'Believe, act, persist, and achieve all you dream of.',

'Turn struggle into strength, and pain into power.',

'Keep learning, keep evolving, keep moving forward.',

'Light shines brightest after the darkest long night.',

'The harder the climb, the better the view feels.',

'Success begins where excuses finally all end now.',

'Courage is action even when the fear still stays.',

'Dreams need effort, faith, and daily persistence.',

'You are built to rise higher than any struggle.',

'Strength grows quietly, even when hope feels lost.',

'Hustle in silence, let success make the noise.',

'Great things take time, trust your steady pace.',

'Discipline outlasts motivation when days feel low.',

'Change starts once you step beyond your comfort.',

'Keep the inner fire alive even in rough storms.',

'Every sunrise whispers hope to start again new.',

'Turn fear to faith, pain to purpose, rise tall.',

'Push limits, embrace growth, and inspire others.',

'Keep moving forward, even when the path feels dark.',

'Strength is built when you refuse to give up now.',

'Your limits exist only if you choose to see them.',

'Stay patient, stay focused, your time will come.',

'Every fall teaches something worth rising for.',

'Hard work today builds a powerful tomorrow soon.',

'Dreams grow stronger when watered with persistence.',

'Let hope guide your heart through every rough day.',

'You are closer than you think, keep pushing on.'
]

In [10]:
roll_numbers = [
'ee23bt055.npy',
'ee23bt031.npy',
'ee23bt040.npy',
'ee23bt047.npy',
'ee23bt050.npy', 
'ee23bt022.npy', 
'ee23bt034.npy', 
'220020010.npy',
'ee23bt032.npy', 
'220020016.npy',
'220020018.npy',
'ee25mr005.npy', 
'220020020.npy',
'ee23bt024.npy', 
'EE23BT060.npy', 
'ee23bt033.npy', 
'ee23bt027.npy', 
'ee23bt017.npy', 
'ee23bt028.npy', 
'ee23bt058.npy', 
'ee23bt051.npy', 
'ee25mt005.npy', 
'ee23bt056.npy', 
'ee25mt016.npy', 
'ee23bt036.npy', 
'ee23bt045.npy' ]

In [12]:
def text_to_bin(text):
    temp_bin_str = ''.join(format(ord(c), '07b') for c in text)
    bin_data = np.array([int(bit) for bit in temp_bin_str], dtype=int)
    return bin_data


def generate_gaussian_noise(signal_length, mean, variance):
    noise = np.random.normal(mean, np.sqrt(variance), signal_length)
    return noise

def bin_to_text(binary_arr):
    bin_str = "".join(str(bit) for bit in binary_arr)
    text = ""
    for i in range(0, len(bin_str), 7):
        byte = bin_str[i:i+7]
        if len(byte) == 7 :
          text += chr(int(byte, 2))
    return text

In [13]:
# roll_arr = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,
#             14,15,16,17,18,19,20,21,22,23,24,25]

# roll_arr = [1, 20, 25]


# for sent_num in roll_arr:
#     # sent_num = 26
#     text = phrases[sent_num]
#     bits = text_to_bin(text)
#     bpsk_symbols = np.array([-1 if bit == 0 else 1 for bit in bits])
#     barker_code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
#     barker_appended_bpsk = np.concatenate((barker_code,  bpsk_symbols))

#     sps = 8
#     zero_padded = np.zeros(len(barker_appended_bpsk) * sps)
#     zero_padded[::sps] = barker_appended_bpsk
    
#     num_taps = 101
#     beta = 0.35
#     Ts = sps # Assume sample rate is 1 Hz, so sample period is 1, so *symbol* period is 8
#     t = np.arange(num_taps) - (num_taps-1)//2
#     h = np.sinc(t/Ts) * np.cos(np.pi*beta*t/Ts) / (1 - (2*beta*t/Ts)**2)

#     ps_conv_output = np.convolve(zero_padded, h, mode = 'full')

#     x_shaped = ps_conv_output[num_taps//2: -1-num_taps//2]

#     tiled = np.tile(x_shaped, 15)

#     rand_num = random.randint(100, 1000)
#     tx_signal = tiled[rand_num:]

#     noise = generate_gaussian_noise(len(tx_signal), 0, 0.2)

#     noisy_signal = tx_signal + noise

#     fs = 1e6 # assume our sample rate is 1 MHz
#     # fo = 10000 # simulate freq offset
#     fo = random.randint(8000, 10000)
#     Ts = 1/fs # calc sample period
#     t = np.arange(0, Ts*len(noisy_signal), Ts) # create time vector
#     rx_signal = noisy_signal * np.exp(1j*2*np.pi*fo*t) # perform freq shift

#     matched_filter =  h
#     MF_op = np.convolve(rx_signal, matched_filter, mode='same')

#     max_allowed = 1
#     rx_max = np.max(np.abs(MF_op))
#     scale_coarse = max_allowed / rx_max
#     normalized_rx = MF_op * scale_coarse

#     squared = normalized_rx**2
#     psd = np.fft.fftshift(np.abs(np.fft.fft(squared)))
#     f = np.linspace(-fs/2.0, fs/2.0, len(psd))

#     max_freq = f[np.argmax(psd)]

#     Ts = 1/fs # calc sample period
#     t = np.arange(0, Ts*len(normalized_rx), Ts) # create time vector
#     coarse_corrected = normalized_rx * np.exp(-1j*2*np.pi*max_freq*t/2.0)

#     power = np.mean(np.abs(coarse_corrected)**2)
#     normalized_2 = coarse_corrected / np.sqrt(power)

#     samples = normalized_2
#     N = len(samples)
#     phase = 0
#     freq = 0
#     # These next two params is what to adjust, to make the feedback loop faster or slower (which impacts stability)
#     alpha = 0.132
#     beta = 0.00932
#     fine_correct = np.zeros(N, dtype=np.complex64)
#     freq_log = []
#     for i in range(N):
#         fine_correct[i] = samples[i] * np.exp(-1j*phase) # adjust the input sample by the inverse of the estimated phase offset
#         error = np.real(fine_correct[i]) * np.imag(fine_correct[i]) # This is the error formula for 2nd order Costas Loop (e.g. for BPSK)

#         # Advance the loop (recalc phase and freq offset)
#         freq += (beta * error)
#         freq_log.append(freq * fs / (2*np.pi)) # convert from angular velocity to Hz for logging
#         phase += freq + (alpha * error)

#         # Optional: Adjust phase so its always between 0 and 2pi, recall that phase wraps around every 2pi
#         while phase >= 2*np.pi:
#             phase -= 2*np.pi
#         while phase < 0:
#             phase += 2*np.pi

#     out = fine_correct[::sps]
#     time_sync = out
#     barker_code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
#     barker_correlated = np.correlate(time_sync, barker_code, mode='full')
#     offset  = np.argmax(np.abs(barker_correlated))+1
#     peak = np.real(barker_correlated)[offset-1]
#     if peak < 0:
#         time_sync = -1*time_sync

#     n_bits = 420  # Number of message bits
#     fr_sync_op = time_sync[offset:offset+n_bits]

#     recovered_bits = (fr_sync_op > 0).astype(int)

#     decoded_text = bin_to_text(recovered_bits)
#     result_prase = decoded_text.split('.')[0] + '.'
#     # print(result_prase)
#     if result_prase == text:
#         print('Decoded text:', sent_num, result_prase)
#         file_name = roll_numbers[sent_num]
#         np.save(file_name, rx_signal)
#     else:
#         print('corrupt:', sent_num)

Decoded text: 0 Dreams grow stronger when watered with persistence.


## Test

In [14]:
for file_num in range(26):
    sps = 8
    num_taps = 101
    beta = 0.35
    Ts = sps # Assume sample rate is 1 Hz, so sample period is 1, so *symbol* period is 8
    t = np.arange(num_taps) - (num_taps-1)//2
    h = np.sinc(t/Ts) * np.cos(np.pi*beta*t/Ts) / (1 - (2*beta*t/Ts)**2)

    fs = 1e6 # assume our sample rate is 1 MHz
    filename = roll_numbers[file_num]
    print(filename)
    rx_signal = np.load(filename)

    matched_filter =  h
    MF_op = np.convolve(rx_signal, matched_filter, mode='same')

    max_allowed = 1
    rx_max = np.max(np.abs(MF_op))
    scale_coarse = max_allowed / rx_max
    normalized_rx = MF_op * scale_coarse

    squared = normalized_rx**2
    psd = np.fft.fftshift(np.abs(np.fft.fft(squared)))
    f = np.linspace(-fs/2.0, fs/2.0, len(psd))

    max_freq = f[np.argmax(psd)]

    Ts = 1/fs # calc sample period
    t = np.arange(0, Ts*len(normalized_rx), Ts) # create time vector
    coarse_corrected = normalized_rx * np.exp(-1j*2*np.pi*max_freq*t/2.0)

    power = np.mean(np.abs(coarse_corrected)**2)
    normalized_2 = coarse_corrected / np.sqrt(power)

    samples = normalized_2
    N = len(samples)
    phase = 0
    freq = 0
    # These next two params is what to adjust, to make the feedback loop faster or slower (which impacts stability)
    alpha = 0.132
    beta = 0.00932
    fine_correct = np.zeros(N, dtype=np.complex64)
    freq_log = []
    for i in range(N):
        fine_correct[i] = samples[i] * np.exp(-1j*phase) # adjust the input sample by the inverse of the estimated phase offset
        error = np.real(fine_correct[i]) * np.imag(fine_correct[i]) # This is the error formula for 2nd order Costas Loop (e.g. for BPSK)

        # Advance the loop (recalc phase and freq offset)
        freq += (beta * error)
        freq_log.append(freq * fs / (2*np.pi)) # convert from angular velocity to Hz for logging
        phase += freq + (alpha * error)

        # Optional: Adjust phase so its always between 0 and 2pi, recall that phase wraps around every 2pi
        while phase >= 2*np.pi:
            phase -= 2*np.pi
        while phase < 0:
            phase += 2*np.pi

    out = fine_correct[::sps]
    time_sync = out
    barker_code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
    barker_correlated = np.correlate(time_sync, barker_code, mode='full')
    offset  = np.argmax(np.abs(barker_correlated))+1
    peak = np.real(barker_correlated)[offset-1]
    if peak < 0:
        time_sync = -1*time_sync

    n_bits = 420  # Number of message bits
    fr_sync_op = time_sync[offset:offset+n_bits]

    recovered_bits = (fr_sync_op > 0).astype(int)

    decoded_text = bin_to_text(recovered_bits)
    result_prase = decoded_text.split('.')[0] + '.'

    print('Decoded text:', file_num, result_prase)